In [1]:
import pandas as pd

In [2]:
import os
from pathlib import Path

In [3]:
PROJECT_DIR = Path(os.environ['PROJECT_DIR'])
PREPROCESSED_DATA_PATH = PROJECT_DIR / 'data/preprocessed_data'

In [4]:
df = pd.read_csv(PREPROCESSED_DATA_PATH / 'data.csv')
split_df = pd.read_csv(PREPROCESSED_DATA_PATH / 'split.csv')

In [5]:
df = df.sample(5, random_state=42)

In [6]:
from typing import Tuple

from tqdm.auto import tqdm
import pandas as pd
from loguru import logger
from torch.utils.data import Dataset
import torch

from src.waveform.utils import load_waveform, trim_waveform


class FineTuningDataset(Dataset):
    """
    A custom Dataset class for fine-tuning models on audio data.

    This class handles the loading and preprocessing of audio waveforms from
    a pandas DataFrame containing metadata such as file paths, start times, 
    and end times. Depending on the `fast_mode` flag, it can either load all 
    waveforms into memory upfront or load them on-the-fly during training.

    Attributes
    ----------
    df : pd.DataFrame
        DataFrame containing metadata for the audio samples.
    filepath_column_name : str
        Name of the column in `df` that contains file paths to the audio files.
    start_time_column_name : str
        Name of the column in `df` that contains start times (in seconds) for trimming the audio.
    end_time_column_name : str
        Name of the column in `df` that contains end times (in seconds) for trimming the audio.
    fast_mode : bool
        If True, all waveforms are loaded into memory during initialization.
    _waveforms : dict
        Dictionary storing preloaded waveforms and their sample rates when `fast_mode` is True.

    Methods
    -------
    __len__()
        Returns the number of samples in the dataset.
    __getitem__(i)
        Returns the i-th sample and its sample rate from the dataset.
    sizeof_fmt(num, suffix="B")
        Converts a byte size into a human-readable format.

    Example
    -------
    >>> dataset = FineTuningDataset(
    ...     df=df,
    ...     filepath_column_name='source',
    ...     start_time_column_name='start_time',
    ...     end_time_column_name='end_time',
    ...     fast_mode=True  # set to False for RAM-optimised data loading
    ... )
    """
    def __init__(
        self, 
        df: pd.DataFrame, 
        filepath_column_name: str,
        target_column_name: str,
        start_time_column_name: str,
        end_time_column_name: str,
        fast_mode: bool,
    ):
        """
        Initializes the FineTuningDataset object.

        If `fast_mode` is set to True, all unique waveforms specified in the 
        DataFrame are preloaded into memory and stored in the `_waveforms` 
        attribute. Otherwise, waveforms are loaded from disk during each 
        call to `__getitem__`.

        Parameters
        ----------
        df : pd.DataFrame
            DataFrame containing metadata for the audio samples.
        filepath_column_name : str
            Name of the column in `df` that contains file paths to the audio files.
        target_column_name : str
            Name of the column in `df` that contains target variable value.
        start_time_column_name : str
            Name of the column in `df` that contains start times (in seconds) for trimming the audio.
        end_time_column_name : str
            Name of the column in `df` that contains end times (in seconds) for trimming the audio.
        fast_mode : bool
            If True, all waveforms are preloaded into memory.

        Returns
        -------
        None
        """
        self.df = df.copy().reset_index(drop=True)
        self.filepath_column_name = filepath_column_name
        self.target_column_name = target_column_name
        self.start_time_column_name = start_time_column_name
        self.end_time_column_name = end_time_column_name
        self.fast_mode = fast_mode
        self._waveforms = {}

        unique_filepaths_cnt = len(self.df[self.filepath_column_name].unique())
        if self.fast_mode:
            logger.info(f'Dataset is initialized in fast mode. All waveforms {unique_filepaths_cnt} will be loaded to RAM.')

            filepaths = self.df[self.filepath_column_name].unique()  # all unique filepaths
            pbar = tqdm(filepaths, desc='Loading waveforms')
            total_size = 0

            for filepath in pbar:
                # load waveform
                waveform, sr = load_waveform(audio_path=filepath)
                self._waveforms[filepath] = (waveform, sr)
                
                # log to progress bar
                total_size += waveform.element_size() * waveform.nelement()
                pbar.set_postfix({'files': len(self._waveforms), 'total_size': self.sizeof_fmt(total_size)})
        else:
            logger.info(
                f'Dataset is initialized in RAM-optimised mode. '
                f'Waveforms ({unique_filepaths_cnt}) will be loaded on-the-fly at each training step.'
            )
        
    def __len__(self):
        """
        Returns the number of samples in the dataset.

        Returns
        -------
        int
            Number of samples in the dataset.
        """
        return len(self.df)

    def __getitem__(self, i: int) -> Tuple[torch.Tensor, int, float]:
        """
        Returns the i-th sample from the dataset.

        This method loads the waveform corresponding to the i-th entry 
        in the DataFrame. If `fast_mode` is enabled, the waveform is 
        retrieved from memory; otherwise, it is loaded from disk. The 
        waveform is then trimmed to the specified start and end times.

        Parameters
        ----------
        i : int
            Index of the sample to retrieve.

        Returns
        -------
        Tuple[torch.Tensor, int, float]
            A tuple containing the waveform as a torch tensor, the sample rate and the target value.
        """
        # get information on audio sample
        row = self.df.iloc[i]
        filepath = row[self.filepath_column_name]
        start_time, end_time = row[self.start_time_column_name], row[self.end_time_column_name]
        target_value = float(row[self.target_column_name])

        # load waveform
        if self.fast_mode:
            waveform, sr = self._waveforms[filepath]
        else:
            waveform, sr = load_waveform(audio_path=filepath)

        # trim waveform into audio sample
        waveform = trim_waveform(
            waveform=waveform, 
            start_time=start_time, 
            end_time=end_time, 
            sample_rate=sr
        )
        return waveform, sr, target_value

    @staticmethod
    def sizeof_fmt(num: int | float, suffix="B") -> str:
        """
        Converts a byte size into a human-readable format.

        Parameters
        ----------
        num : int
            The size in bytes.
        suffix : str, optional
            The suffix to append to the formatted size (default is "B").

        Returns
        -------
        str
            Human-readable string representation of the size.
        """
        for unit in ("", "Ki", "Mi", "Gi", "Ti", "Pi", "Ei", "Zi"):
            if abs(num) < 1024.0:
                return f"{num:3.1f}{unit}{suffix}"
            num /= 1024.0
        return f"{num:.1f}Yi{suffix}"

## RAM-optimised mode

In [7]:
dataset = FineTuningDataset(
    df=df,
    filepath_column_name='source',
    target_column_name='phq_score',
    start_time_column_name='start_time',
    end_time_column_name='end_time',
    fast_mode=False
)

2024-09-13 09:07:10.377 | INFO     | __main__:__init__:116 - Dataset is initialized in RAM-optimised mode. Waveforms (5) will be loaded on-the-fly at each training step.


In [8]:
dataset._waveforms

{}

In [9]:
dataset[0]

(tensor([[ 12,  11,  10,  ..., 182, 183, 185]], dtype=torch.int16),
 48000,
 19.0)

## Fast mode

In [10]:
dataset = FineTuningDataset(
    df=df,
    filepath_column_name='source',
    target_column_name='phq_score',
    start_time_column_name='start_time',
    end_time_column_name='end_time',
    fast_mode=True
)

2024-09-13 09:07:59.863 | INFO     | __main__:__init__:101 - Dataset is initialized in fast mode. All waveforms 5 will be loaded to RAM.


Loading waveforms:   0%|          | 0/5 [00:00<?, ?it/s]

In [11]:
dataset._waveforms

{'/workspace/speech-based-distress-recognition/data/raw_data/698_P/698_AUDIO.wav': (tensor([[0, 0, 0,  ..., 6, 4, 4]], dtype=torch.int16),
  48000),
 '/workspace/speech-based-distress-recognition/data/raw_data/697_P/697_AUDIO.wav': (tensor([[ -1, -10, -29,  ..., -47, -46, -46]], dtype=torch.int16),
  48000),
 '/workspace/speech-based-distress-recognition/data/raw_data/636_P/636_AUDIO.wav': (tensor([[-94, -96, -96,  ...,  49,  70, 107]], dtype=torch.int16),
  16000),
 '/workspace/speech-based-distress-recognition/data/raw_data/321_P/321_AUDIO.wav': (tensor([[   18,    19,    17,  ..., -1752, -1634, -1519]], dtype=torch.int16),
  16000),
 '/workspace/speech-based-distress-recognition/data/raw_data/691_P/691_AUDIO.wav': (tensor([[  0,   0,   1,  ..., -45, -40, -35]], dtype=torch.int16),
  48000)}

In [12]:
dataset[0]

(tensor([[ 12,  11,  10,  ..., 182, 183, 185]], dtype=torch.int16),
 48000,
 19.0)